In [22]:
import numpy as np 
import scipy.linalg 
import matplotlib.pyplot as plt

In [23]:
class VectorizedStochasticGridWorld:
    """
    A configurable slippery Grid-World environment.
    Builds the transition tensor P (num_states, num_actions, num_states)
    and reward table R (num_states, num_actions).
    """
 
    def __init__(self, height=10, width=10, slip_prob=0.15, step_cost=-0.1,
                 goal_reward=10.0, trap_penalty=-10.0,
                 goal_state=None, trap_state=None):
        self.height = height
        self.width = width
        self.num_states = height * width
        self.num_actions = 4  # 0: Up, 1: Down, 2: Left, 3: Right
 
        # Default goal = bottom-right corner, default trap = center tile
        self.goal_state = goal_state if goal_state is not None else self.num_states - 1
        self.trap_state = trap_state if trap_state is not None else (
            (height // 2) * width + (width // 2)
        )
 
        self.slip_prob = slip_prob
        self.step_cost = step_cost
        self.goal_reward = goal_reward
        self.trap_penalty = trap_penalty
 
        self.P = np.zeros((self.num_states, self.num_actions, self.num_states))
        self.R = np.zeros((self.num_states, self.num_actions))
        self.build_world()
 
    def build_world(self):
        # Total slip probability is split evenly across the 3 unintended directions
        slip_each = self.slip_prob / (self.num_actions - 1)
        intended_prob = 1.0 - self.slip_prob
 
        for s in range(self.num_states):
            if s == self.goal_state or s == self.trap_state:
                # Terminal states: absorbing loop
                for a in range(self.num_actions):
                    self.P[s, a, s] = 1.0
                continue
 
            r, c = divmod(s, self.width)
            # Map out coordinates if we went Up, Down, Left, or Right
            neighbors = {
                0: (max(r - 1, 0), c),                      # Up (prevent leaving top wall)
                1: (min(r + 1, self.height - 1), c),        # Down (prevent leaving bottom wall)
                2: (r, max(c - 1, 0)),                       # Left (prevent leaving left wall)
                3: (r, min(c + 1, self.width - 1))           # Right (prevent leaving right wall)
            }
 
            for a in range(self.num_actions):
                # Target state for the intended action
                intended_r, intended_c = neighbors[a]
                intended_state = intended_r * self.width + intended_c
 
                # Assign probability to intended move
                self.P[s, a, intended_state] += intended_prob
 
                # Assign remaining probability evenly across slip directions
                for slip_action in range(self.num_actions):
                    if slip_action != a:
                        sr, sc = neighbors[slip_action]
                        slip_state = sr * self.width + sc
                        self.P[s, a, slip_state] += slip_each
 
                # Define Reward: Goal -> +reward, Trap -> +penalty, else step cost
                for s_next in range(self.num_states):
                    prob = self.P[s, a, s_next]
                    if prob > 0:
                        if s_next == self.goal_state:
                            self.R[s, a] += prob * self.goal_reward
                        elif s_next == self.trap_state:
                            self.R[s, a] += prob * self.trap_penalty
                        else:
                            self.R[s, a] += prob * self.step_cost
 
 
class DynamicProgrammingSolver:
    """
    Solves a given GridWorld environment using Value Iteration
    and exact (SciPy-based) Policy Iteration.
    """
 
    def __init__(self, env, gamma=0.99):
        self.env = env
        self.gamma = gamma
 
    def value_iteration(self, theta=1e-8, max_iters=1000):
        """
        Run the iterative loop to find the optimal values and directions.
        Returns (V, optimal_policy, delta_history).
        """
        V = np.zeros(self.env.num_states)
        history = []
 
        for i in range(max_iters):
            # Vectorized Q-value update: sums over next_state automatically
            Q = self.env.R + self.gamma * np.tensordot(self.env.P, V, axes=(2, 0))
 
            # Best score among the actions for each state
            V_new = np.max(Q, axis=1)
 
            delta = np.max(np.abs(V_new - V))
            history.append(delta)
            V = V_new
 
            if delta < theta:
                print(f"Value Iteration converged in {i + 1} iterations! (gamma={self.gamma})")
                break
 
        final_Q = self.env.R + self.gamma * np.tensordot(self.env.P, V, axes=(2, 0))
        optimal_policy = np.argmax(final_Q, axis=1)
        return V, optimal_policy, history
 
    def policy_iteration(self, max_iters=100):
        """
        Run Policy Iteration using SciPy's exact linear system solver.
        Returns (V, optimal_policy).
        """
        policy = np.zeros(self.env.num_states, dtype=int)
        V = np.zeros(self.env.num_states)
        state_indices = np.arange(self.env.num_states)
 
        for i in range(max_iters):
            # 1. Exact Policy Evaluation: solve (I - gamma * P_pi) * V = R_pi
            P_pi = self.env.P[state_indices, policy, :]
            R_pi = self.env.R[state_indices, policy]
 
            A = np.eye(self.env.num_states) - self.gamma * P_pi
            V = scipy.linalg.solve(A, R_pi)
 
            # 2. Policy Improvement
            Q = self.env.R + self.gamma * np.tensordot(self.env.P, V, axes=(2, 0))
            new_policy = np.argmax(Q, axis=1)
 
            if np.array_equal(new_policy, policy):
                print(f"Policy Iteration converged in {i + 1} steps! (gamma={self.gamma})")
                break
            policy = new_policy
 
        return V, policy
 
 
if __name__ == "__main__":
    # Initialize the stochastic Grid-World
    env = VectorizedStochasticGridWorld(height=10, width=10, slip_prob=0.15, step_cost=-0.1)
 
    # Analyze multiple discount factors gamma
    discount_factors = [0.9, 0.95, 0.99, 0.999]
 
    for gamma in discount_factors:
        print(f"\n--- Running Dynamic Programming for Gamma = {gamma} ---")
        solver = DynamicProgrammingSolver(env, gamma=gamma)
 
        # Run Value Iteration
        v_vi, pi_vi, hist_vi = solver.value_iteration()
        # Run Policy Iteration
        v_pi, pi_pi = solver.policy_iteration()
 
        # Verify that Value Iteration and Policy Iteration yield mathematically identical values
        max_diff = np.max(np.abs(v_vi - v_pi))
        print(f"Max Value discrepancy between VI and PI: {max_diff:.2e}")
 
    # Render the optimal path topology
    solver = DynamicProgrammingSolver(env, gamma=0.99)
    optimal_values, optimal_policy = solver.policy_iteration()
 
    # Plot final State Values Heatmap
    plt.figure(figsize=(8, 6))
    plt.imshow(optimal_values.reshape((10, 10)), cmap="viridis", origin="upper")
    plt.colorbar(label="Expected Value $V(s)$")
    plt.title(r"Vectorized Dynamic Programming: Optimal Value Grid ($\gamma=0.99$)")
 
    # Render Policy arrows
    arrows = {0: "↑", 1: "↓", 2: "←", 3: "→"}
    for r in range(10):
        for c in range(10):
            idx = r * 10 + c
            if idx == env.goal_state:
                label = "G"
            elif idx == env.trap_state:
                label = "T"
            else:
                label = arrows[optimal_policy[idx]]
            plt.text(c, r, label, ha="center", va="center",
                      color="white" if optimal_values[idx] < 0 else "black",
                      fontweight="bold")
 
    print("\nState Values visual plot saved as 'mdp_optimal_values.png'.")
    plt.savefig("mdp_optimal_values.png")
    plt.close()


--- Running Dynamic Programming for Gamma = 0.9 ---
Value Iteration converged in 52 iterations! (gamma=0.9)
Max Value discrepancy between VI and PI: 8.68e-09

--- Running Dynamic Programming for Gamma = 0.95 ---
Value Iteration converged in 57 iterations! (gamma=0.95)
Max Value discrepancy between VI and PI: 8.67e-09

--- Running Dynamic Programming for Gamma = 0.99 ---
Value Iteration converged in 61 iterations! (gamma=0.99)
Max Value discrepancy between VI and PI: 1.19e-08

--- Running Dynamic Programming for Gamma = 0.999 ---
Value Iteration converged in 62 iterations! (gamma=0.999)
Max Value discrepancy between VI and PI: 1.20e-08

State Values visual plot saved as 'mdp_optimal_values.png'.
